# Methods to solve system of equations

### 1.	Write a python program to implement Gaussian elimination method to solve a system of linear equations.

In [2]:
import numpy as np
from sympy import Symbol
from math import isclose

def echelon_form(mat, pivots=False):
    mat = np.array(mat, dtype=float)
    r, c = mat.shape
    pivot_set = set()
    shift = 0
    
    for i in range(min(r, c)):
        while i+shift<c and all(isclose(elem, 0, abs_tol=1e-08)
                                for elem in mat[i:, i+shift]): shift += 1
        if i+shift==c: break
        
        pivot = mat[i, i+shift]
        for next_row in range(i+1, r):
            below_pivot = mat[next_row, i+shift]
            if not isclose(below_pivot, 0, abs_tol=1e-08):
                if not isclose(pivot, 0, abs_tol=1e-08):
                    mat[next_row] -= (below_pivot/pivot)*mat[i]
                else:
                    mat[[i, next_row]] = mat[[next_row, i]]
                    pivot = below_pivot
        
        pivot_set.add(i+shift)
        mat[i, :i+shift] = 0.
    
    if pivots:
        return mat, pivot_set
    return mat

def solve(mat):
    mat, pivots = echelon_form(mat, pivots=True)
    r, c = mat.shape
    
    if c-1 in pivots:
        return False
    
    A, B = mat[len(pivots)-1::-1, -2::-1], mat[len(pivots)-1::-1, -1]
    x = []
    
    for eq, val in zip(A, B):
        sm = 0
        for i, coeff in enumerate(eq):
            if i==len(x):
                if c-2-i in pivots:
                    x.append((val-sm)/coeff)
                    break
                x.append(Symbol(f'x{c-1-i}'))
            sm += coeff*x[i]
    
    for i in range(c-1-len(x), 0, -1):
        x.append(Symbol(f'x{i}'))
    
    return x[::-1]

In [3]:
def input_mat(square=False, dtype=float):
    if square:
        r = c = int(input('Enter the dimension of the square matrix: '))
    else:
        r = int(input('Enter the number of rows: '))
        c = int(input('Enter the number of columns: '))
    
    ret_mat = []
    for i in range(r):
        ret_row = [dtype(input(f'Enter element for position [{i+1}, {j+1}]: ')) for j in range(c)]
        ret_mat.append(ret_row)
    
    return ret_mat

A = input_mat()
print('Augmented matrix:', A)

print('Solution:', solve(A))

Augmented matrix: [[1.0, 1.0, 1.0, 6.0], [1.0, -1.0, 1.0, 2.0], [1.0, -1.0, -1.0, -4.0]]
Solution: [1.0, 2.0, 3.0]


### 2.	Write a python program to implement Cramer’s rule to solve a system of linear equations.

In [3]:
def transpose(mat):
    r, c = len(mat), len(mat[0])
    
    ret_mat = []
    for j in range(c):
        ret_row = [row[j] for row in mat]
        ret_mat.append(ret_row)
    
    return ret_mat

def det(mat):
    r, c = len(mat), len(mat[0])
    
    if r!=c:
        raise ValueError('matrix must be square')
    if r==1==c:
        return mat[0][0]
    if r==2==c:
        return (mat[0][0]*mat[1][1]) - (mat[0][1]*mat[1][0])
    
    row1, rest_rows_T = mat[0], transpose(mat[1:])
    
    val = 0
    for i, elem in enumerate(row1):
        rem_row_col = transpose([rest_rows_T[k] for k in range(r) if k!=i])
        val += (-1)**i * elem * det(rem_row_col)
    
    return val

def cramers(A, B):
    det_A = det(A)
    if det_A==0:
        return
    
    x = []
    for i in range(len(A)):
        Ai = transpose(A)
        Ai[i] = B
        det_Ai = det(Ai)
        
        x.append(det_Ai/det_A)
    
    return x

In [5]:
def input_mat(square=False, dtype=float):
    if square:
        r = c = int(input('Enter the dimension of the square matrix: '))
    else:
        r = int(input('Enter the number of rows: '))
        c = int(input('Enter the number of columns: '))
    
    ret_mat = []
    for i in range(r):
        ret_row = [dtype(input(f'Enter element for position [{i+1}, {j+1}]: ')) for j in range(c)]
        ret_mat.append(ret_row)
    
    return ret_mat

def input_vec(dtype=float):
    dim = int(input('Enter dimension of vector: '))
    return [dtype(input(f'Enter element {i+1}: ')) for i in range(dim)]

A = input_mat(square=True)
B = input_vec()
print('Matrix A:', A)
print('Vector B:', B)

print('Solution Ax=B:', cramers(A, B))

Matrix A: [[1.0, 1.0, 1.0], [1.0, -1.0, 1.0], [1.0, -1.0, -1.0]]
Vector B: [6.0, 2.0, -4.0]
Solution Ax=B: [1.0, 2.0, 3.0]


### 3.	Write a python program to accept a system of linear equations from a user and find its solution. (any of the above method)